In [ ]:
import numpy as np
import torch
from torchvision import datasets, transforms
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from optimizer import OrthogonalOptimizer
from linearb import LinearB
from convb import ConvB
import gc
import os

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset  = datasets.MNIST(root='./data', train=False, transform=transform, download=True)

train_loader = DataLoader(dataset=train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(dataset=test_dataset, batch_size=1000, shuffle=False)

In [4]:
# class mnist(nn.Module):
#     def __init__(self, num_bases):
#         super().__init__()
#         q, _ = torch.linalg.qr(torch.randn(784, 784, device=device))
#         self.register_buffer('q', q)
#         self.layer1 = LinearB(784, 1024, num_bases=num_bases)
#         self.relu1 = nn.ReLU()
#         self.drop1 = nn.Dropout(0.2)
#         self.layer2 = LinearB(1024, 1024, num_bases=num_bases)
#         self.relu2 = nn.ReLU()
#         self.classifier = nn.Linear(1024, 10)
        
#     def forward(self, x):
#         x = x.flatten(start_dim=1)
#         x = x @ self.q
#         x = self.relu1(self.layer1(x))
#         residual = x
#         x = self.drop1(x)
#         x = self.relu2(self.layer2(x))
#         x = x + residual
#         return self.classifier(x)

class mnist(nn.Module):
    def __init__(self, num_bases):
        super().__init__()
        q, _ = torch.linalg.qr(torch.randn(784, 784, device=device))
        self.register_buffer('q', q)
        self.layer1 = ConvB(1, 1024, kernel_size=3, num_bases=num_bases, padding=1)
        self.relu1 = nn.ReLU()
        self.drop1 = nn.Dropout(0.2)
        self.layer2 = ConvB(1024, 1024, kernel_size=3, num_bases=num_bases, padding=1)
        self.relu2 = nn.ReLU()
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(1024, 10)
        
    def forward(self, x):
        batch_size = x.size(0)
        x = x.flatten(start_dim=1)
        x = x @ self.q
        x = x.view(batch_size, 1, 28, 28)
        x = self.relu1(self.layer1(x))
        residual = x
        x = self.drop1(x)
        x = self.relu2(self.layer2(x))
        x = x + residual
        x = self.pool(x)
        x = x.flatten(start_dim=1)
        return self.classifier(x)

model = mnist(num_bases=3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = OrthogonalOptimizer(model.parameters(), lr=0.01, momentum=0.9, num_flips=5)
gc.collect()

In [ ]:
epochs = 5
train_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []
checkpoint_path = "mnist_checkpoint.pt"
start_epoch = 0

In [ ]:
if os.path.exists(checkpoint_path):
    print(f"Found checkpoint '{checkpoint_path}', resuming training")
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    train_losses = checkpoint['train_losses']
    train_accuracies = checkpoint['train_accuracies']
    test_losses = checkpoint['test_losses']
    test_accuracies = checkpoint['test_accuracies']
    print(f"Resuming from loop index {start_epoch}, epoch {start_epoch + 1}")
else:
    print("No checkpoint found, starting training from scratch.")

In [ ]:
for epoch in range(start_epoch, epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
    print(f'Epoch [{epoch + 1}/{epochs}]')
    print("Training loss", (running_loss / len(train_loader)))  
    train_losses.append(running_loss / len(train_loader))
    print("Training accuracy", (correct *100/ total),'%')
    train_accuracies.append(correct * 100 / total)

    model.eval()
    running_loss_eval = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss_eval += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print("Testing loss", (running_loss_eval / len(test_loader)))
    test_losses.append(running_loss_eval / len(test_loader))
    print("Testing accuracy", (correct*100 / total),'%')
    test_accuracies.append(correct * 100 / total)

    checkpoint_data = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_losses': train_losses,
        'train_accuracies': train_accuracies,
        'test_losses': test_losses,
        'test_accuracies': test_accuracies
    }
    tmp_checkpoint_path = checkpoint_path + ".tmp"
    torch.save(checkpoint_data, tmp_checkpoint_path)
    os.replace(tmp_checkpoint_path, checkpoint_path)
    print(f"Checkpoint saved securely at end of epoch {epoch + 1}")
    
    gc.collect()